In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader, random_split

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

from DatasetLoader import DatasetLoader
from CCN1D import CCN1D
from Transformer import Transformer

In [41]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 15
TRAIN_RATIO = 0.8
PATIENCE = 5

In [42]:
class CustomDataset(data.Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.drop(columns=[col for col in ["Battery", "Cell"] if col in dataframe.columns])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        features = self.dataframe.iloc[idx, :-1].values.astype(np.float32)
        label = float(self.dataframe.iloc[idx, -1])
        return torch.tensor(features), torch.tensor(label, dtype=torch.float32)

In [43]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='best_model.pt'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Salva il modello quando la validation loss diminuisce.'''
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [44]:
from sklearn.model_selection import train_test_split

# Caricamento dati
df = pd.read_csv("synthetic_data.csv")

# 1. Split iniziale: separazione in Train + Val e Test (es. 10% per il test finale)
train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# 2. Split secondario: separazione in Train e Val
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)

# 3. Scaling dei dati (Fit solo sul TRAIN)
# Crea uno scaler separato per la Y
scaler_x = StandardScaler()
scaler_y = StandardScaler()

features = [col for col in df.columns if col not in ["RUL", "Battery", "Cell"]]

# Fit degli scaler solo sul set di TRAIN per evitare leakage
scaler_x.fit(train_df[features])
scaler_y.fit(train_df[["RUL"]]) # Nota le doppie parentesi per mantenere il formato 2D

def scale_dataframe(df_in, sc_x, sc_y, feature_cols):
    df_out = df_in.copy()
    # Scala le feature
    df_out[feature_cols] = sc_x.transform(df_in[feature_cols])
    # Scala la target RUL
    df_out["RUL"] = sc_y.transform(df_in[["RUL"]])
    return df_out

train_df_scaled = scale_dataframe(train_df, scaler_x, scaler_y, features)
val_df_scaled = scale_dataframe(val_df, scaler_x, scaler_y, features)
test_df_scaled = scale_dataframe(test_df, scaler_x, scaler_y, features)

# 4. Creazione Dataset e Dataloader
train_dataset = CustomDataset(train_df_scaled)
val_dataset = CustomDataset(val_df_scaled)
test_dataset = CustomDataset(test_df_scaled)

trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
valloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
testloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = CCN1D(input_channels=9, hidden_channels=1024, num_layers=4, dropout=0.3)

In [46]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    preds, labels = [], []

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        inputs = inputs.unsqueeze(2)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        preds.append(outputs.detach().cpu().numpy())
        labels.append(targets.cpu().numpy())

    preds = np.concatenate(preds)
    labels = np.concatenate(labels)

    avg_loss = total_loss / len(dataloader.dataset) 
    avg_r2 = r2_score(labels, preds)
    return avg_loss, avg_r2


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    preds, labels = [], []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            inputs = inputs.unsqueeze(2)

            outputs = model(inputs).squeeze()
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)

            preds.append(outputs.cpu().numpy())
            labels.append(targets.cpu().numpy())

    preds = np.concatenate(preds)
    labels = np.concatenate(labels)
    avg_loss = total_loss / len(dataloader.dataset)
    avg_r2 = r2_score(labels, preds)
    return avg_loss, avg_r2

In [47]:
def test(model, dataloader):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0
    preds, labels = [], []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            inputs = inputs.unsqueeze(2)
            #targets = targets.unsqueeze(1)

            if torch.isnan(inputs).any():
                print("NaN in input, skipping batch.")
                continue

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)

            preds.append(outputs.cpu().numpy())
            labels.append(targets.cpu().numpy())

    preds = np.concatenate(preds)
    labels = np.concatenate(labels)
    avg_loss = total_loss / len(dataloader.dataset)
    avg_r2 = r2_score(labels, preds)

    return avg_loss, avg_r2

In [48]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
early_stopper = EarlyStopping(patience=PATIENCE, verbose=True, path='checkpoint_rul.pth')

print("--- Inizio Addestramento ---")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_r2 = train_one_epoch(model, trainloader, criterion, optimizer, DEVICE)
    val_loss, val_r2 = validate(model, valloader, criterion, DEVICE)

    print(f'Epoch {epoch:03d}: | Train Loss: {train_loss:.4f} | Train R2: {train_r2:.4f} | Val Loss: {val_loss:.4f} | Val R2: {val_r2:.4f}')

    early_stopper(val_loss, model)
    if early_stopper.early_stop:
        print("Early stopping triggered")
        break

# CARICA IL MIGLIOR MODELLO SALVATO
model.load_state_dict(torch.load('checkpoint_rul.pth'))
print("Best model loaded.")

--- Inizio Addestramento ---


RuntimeError: Given groups=1, weight of size [1024, 8, 3], expected input[64, 9, 1] to have 8 channels, but got 9 channels instead

In [ ]:
final_loss, final_r2 = validate(model, testloader, criterion, DEVICE)

In [ ]:
final_loss

0.004766573663491956

In [ ]:
final_r2

0.9950780868530273